# 01 · Create the medallion tables

This lab reworks chapter 03 of *Practical Data Engineering with Apache Projects*
onto the kind stack: **Postgres + SeaweedFS** as sources, **Iceberg** (REST
catalog `demo`, default catalog) as the lakehouse.

We model three layers:

| Layer | Meaning | Tables |
|---|---|---|
| **bronze** | raw copies of the sources | `users`, `items`, `purchases`, `pageviews` |
| **silver** | validated + enriched | `users`, `items`, `purchases_enriched`, `pageviews_by_items` |
| **gold** | analytics-ready | `item_performance` |

This notebook just creates the empty tables. Run `02` → `03` → `04` to fill them.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("01-create-tables").getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version, "| default catalog:", spark.conf.get("spark.sql.defaultCatalog"))

## Namespaces

In [ ]:
for ns in ("bronze", "silver", "gold"):
    spark.sql(f"CREATE NAMESPACE IF NOT EXISTS demo.{ns}")
spark.sql("SHOW NAMESPACES IN demo").show()

## Bronze — raw ingest targets

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS demo.bronze.users (
    id BIGINT, first_name STRING, last_name STRING, email STRING,
    created_at TIMESTAMP, updated_at TIMESTAMP
) USING iceberg PARTITIONED BY (days(created_at))
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS demo.bronze.items (
    id BIGINT, name STRING, category STRING, price DECIMAL(7,2),
    inventory INT, created_at TIMESTAMP, updated_at TIMESTAMP
) USING iceberg PARTITIONED BY (category)
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS demo.bronze.purchases (
    id BIGINT, user_id BIGINT, item_id BIGINT, quantity INT,
    purchase_price DECIMAL(12,2), created_at TIMESTAMP, updated_at TIMESTAMP
) USING iceberg PARTITIONED BY (days(created_at))
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS demo.bronze.pageviews (
    user_id BIGINT, url STRING, channel STRING, received_at TIMESTAMP
) USING iceberg PARTITIONED BY (days(received_at))
""")
print("bronze tables ready")

## Silver — validated + enriched

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS demo.silver.users (
    id BIGINT, first_name STRING, last_name STRING, email STRING,
    created_at TIMESTAMP, updated_at TIMESTAMP,
    valid_email BOOLEAN, full_name STRING
) USING iceberg PARTITIONED BY (days(created_at))
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS demo.silver.items (
    id BIGINT, name STRING, category STRING, price DECIMAL(7,2),
    inventory INT, created_at TIMESTAMP, updated_at TIMESTAMP
) USING iceberg PARTITIONED BY (category)
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS demo.silver.purchases_enriched (
    id BIGINT, user_id BIGINT, item_id BIGINT, quantity INT,
    purchase_price DECIMAL(12,2), total_price DECIMAL(14,2),
    user_email STRING, item_name STRING, item_category STRING,
    purchase_date DATE, purchase_hour INT,
    created_at TIMESTAMP, updated_at TIMESTAMP
) USING iceberg PARTITIONED BY (days(created_at))
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS demo.silver.pageviews_by_items (
    user_id BIGINT, item_id BIGINT, page STRING,
    item_name STRING, item_category STRING, channel STRING,
    received_at TIMESTAMP
) USING iceberg PARTITIONED BY (days(received_at))
""")
print("silver tables ready")

## Gold — analytics

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS demo.gold.item_performance (
    item_id BIGINT, item_name STRING, item_category STRING,
    items_sold BIGINT, orders BIGINT, revenue DECIMAL(20,2),
    pageviews BIGINT, conversion_rate DOUBLE
) USING iceberg PARTITIONED BY (item_category)
""")
print("gold table ready")

## Verify

In [ ]:
for ns in ("bronze", "silver", "gold"):
    print(f"demo.{ns}:")
    spark.sql(f"SHOW TABLES IN demo.{ns}").show()